# GRPO Training — Step 1: simple tool-use reward

Offline GRPO (Group Relative Policy Optimization) for a Polymarket trading
agent on Google Colab, using QLoRA (4-bit base + LoRA adapters) with vLLM for
rollouts.

**Reward (step 1 of the training plan):** +1 if the agent called the
`search_x_twitter` tool during the rollout, -1 otherwise. Later steps replace
this with the realized-PnL reward (see README.md / HISTORY.md at the repo root).

## Workflow

1. **Set up the agent** — system prompt + personas + 9 inlined MCP tools
2. **Generate traces with vLLM** — LoRA adapter enabled, per-token `log_probs` captured
3. **Compute reference probs** — base model, via vLLM `prompt_logprobs`
4. **Reward** — tool-use check (`search_x_twitter`)
5. **Offline GRPO** — TRL `GRPOTrainer` with QLoRA and a custom `rollout_func`
   (generation happens in step 2; training itself is offline)

Run the cells top to bottom, then `await _rl_loop()`.

## Notes

- Markets rotate per rollout; a fixed held-out set is reserved for eval.
- Adapters are chained rollout-to-rollout and mirrored to Google Drive so runs
  resume across Colab VM restarts.
- Development/debug cells (TEST-A/B2/C, overfit smoke test, crash-recovery
  resume, Drive race tests, adapter diagnostics) live in
  `archive/simple_reward_debug.ipynb`.

In [ ]:
# Cell 1: Install dependencies & imports
# Run this first in Google Colab
#
# vLLM 0.29.0 pins torch==2.13.0 and torchvision==0.28.0 exactly.
# We install those first so pip's resolver is satisfied, then the rest.
# If you change the vLLM version, update these pins to match.

!pip uninstall -y torchaudio
!pip install -q torch==2.13.0 torchvision==0.28.0
!pip install -q vllm==0.31.0 trl==1.15.0 datasets peft==0.21.1 transformers==5.17.0 accelerate bitsandbytes ddgs
print('IMPORTANT: torch was just upgraded, restart the kernel now (Runtime > Restart session) and re-run from Cell 1, or torchvision ops will stay broken.')


In [ ]:
import json
import re
import os
import time
import asyncio
import requests
from pathlib import Path
from datetime import UTC, datetime
from typing import Any

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

print(f'torch: {torch.__version__}')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')

# Suppress noisy TLS close_notify warnings from primp/h2 (Polymarket API)
import logging
logging.getLogger('primp.async_impl.h2_client.pool').setLevel(logging.ERROR)

In [ ]:
# Cell 2c: Mount Google Drive (REQUIRED for adapter mirroring + resume across VMs)
# Run this cell YOURSELF once per session BEFORE the training loops.
# A Google sign-in popup will appear in this notebook: approve it and paste the code back if asked.
# I cannot run it for you because the approval must happen in your browser.
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted. Adapters will mirror to: /content/drive/MyDrive/RL_supply_disk/grpo_output')


In [ ]:
# Cell 2: Configuration
# TEST MODE: 4B model for pipeline validation (fits local disk + 40GB GPU). Switch back to 27B later.

# --- Models ---
VLLM_MODEL = 'cyankiwi/Qwen3.5-4B-AWQ-BF16-INT4'
TRAINING_MODEL = 'Qwen/Qwen3.5-4B'
LORA_ADAPTER_PATH = None

# --- Market data ---
MARKETS_JSON = 'top_k_markets_2026-10-14.json'
MARKETS_URL = None

# --- Generation ---
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 4096
MAX_COMPLETION_LENGTH = 50000
MAX_TOOL_ROUNDS = 10
MAX_PARSE_RETRIES = 2
MAX_CONCURRENT_GENERATION_AGENTS = 46
NB_WORKERS_GENERATION = 50 # Agents generating tokens (MAX_CONCURRENT_GENERATION_AGENTS) while others executing tools

# --- Rollout / training batches ---
# Dataset has 1 row per market; each row yields NUM_GENERATIONS completions.
NUM_GENERATIONS = 5            # G=5 (5 personas per market)
PER_DEVICE_BATCH_SIZE = 2      # micro-batch: 2 generations
GRAD_ACCUM_STEPS = 25          # 25 micro-steps per optimizer step so 1 updates every 50 generations (every 10 markets)
MARKETS_PER_ROLLOUT = 100    # loop rotation unit (rotate through training markets in chunks of 100)
NUM_ITERATIONS = 2             # mu=2 (number of epochs over the same group of training markets (rollout))
LEARNING_RATE = 1e-5
BETA = 0.0
EPSILON = 0.2
LOSS_TYPE = 'dapo'
SCALE_REWARDS = 'group'
OUTPUT_DIR = '/content/grpo_output'

# --- LoRA config for training ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'v_proj']

print('vLLM model: ' + VLLM_MODEL)
print('training model: ' + TRAINING_MODEL)
print('pool file: ' + MARKETS_JSON)
print('EFFECTIVE batch per optimizer step: ' + str(PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
print('Completions per optimizer step: ' + str(PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS * NUM_GENERATIONS))
print('max concurrent generation agents: ' + str(MAX_CONCURRENT_GENERATION_AGENTS))


In [ ]:
# Cell 2b: Weights & Biases setup for GPU & system metrics
# Modified: non-interactive for Colab MCP automation

try:
    import wandb
except ImportError:
    !pip install -q wandb
    import wandb

import os
import getpass

# Authenticate — try Colab secrets, fall back to disabled mode
if not os.environ.get('WANDB_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
        print('W&B API key loaded from Colab secrets.')
    except Exception:
        print('No W&B API key found. Running W&B in disabled mode.')
        os.environ['WANDB_MODE'] = 'disabled'

WANDB_PROJECT = 'grpo-trading-rl'
WANDB_RUN_NAME = f'grpo-investigation-{time.strftime("%Y%m%d-%H%M%S")}'

wandb.init(
    project=WANDB_PROJECT,
    settings=wandb.Settings(
        _disable_stats=False,
        _disable_meta=False,
    ),
    name=WANDB_RUN_NAME,
    config={
        'vllm_model': VLLM_MODEL,
        'training_model': TRAINING_MODEL,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_new_tokens': MAX_NEW_TOKENS,
        'max_tool_rounds': MAX_TOOL_ROUNDS,
        'num_generations': NUM_GENERATIONS,
        'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'learning_rate': LEARNING_RATE,
        'beta': BETA,
        'epsilon': EPSILON,
        'loss_type': LOSS_TYPE,
        'max_completion_length': MAX_COMPLETION_LENGTH,
        'markets_per_rollout': MARKETS_PER_ROLLOUT,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
    },
)

print(f'W&B run initialized: {WANDB_PROJECT}/{WANDB_RUN_NAME}')
print(f'  Dashboard: {wandb.run.get_url()}')
print(f'  GPU detected: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Cell 3a: System prompt and personas
BASE_PROMPT = (
    "You are a professional algorithmic trader operating on Polymarket, "
    "a prediction-market platform.\n\n"
    "## Your task\n"
    "You are given a market (condition_id, question, description, outcomes, end_date).\n"
    "You must produce exactly **ONE trading strategy** for this market.\n\n"
    "## Constraints\n"
    "- You can only **BUY** tokens (YES or NO). They will be **sold automatically\n"
    "  24 hours later** at whatever the market price is at that time.\n"
    "- Your goal is to maximize profit — or at minimum avoid losing money —\n"
    "  over that 24-hour window.\n"
    "- ``bet_amount_usd`` is the dollar amount you want to deploy.\n"
    "  It must not exceed the available order-book liquidity (check the\n"
    "  ORDER BOOK TOOLS to see how much depth is available).\n"
    "- ``confidence_score`` (0-100) reflects your conviction — it is\n"
    "  informational only and does NOT directly control bet sizing.\n"
    "- If you are not confident enough, you may **PASS** (no trade).\n\n"
    "## Available tools\n"
    "You have access to tools that let you:\n"
    "- Inspect the current order book (midpoint prices, spreads, depth).\n"
    "- Check liquidity depth and taker-fee schedules.\n"
    "- Search the web for news and facts.\n"
    "- Search X/Twitter for real-time sentiment.\n"
    "- Calculate expected value and run arithmetic.\n\n"
    "**Use these tools to gather key information, but do NOT over-research.**\n"
    "## Output format (MANDATORY)\n"
    "Output a single JSON object — no extra commentary:\n\n"
    "```json\n"
    "{{\n"
    '  "strategy_name": "Short label",\n'
    '  "reasoning": "Why you chose this trade (2-5 sentences).",\n'
    '  "final_decision": "BUY YES",\n'
    '  "token_id": "<token_id for YES or NO from the market data>",\n'
    '  "bet_amount_usd": <dollar amount, capped at visible liquidity>,\n'
    '  "confidence_score": <0-100, conviction level>\n'
    "}}\n"
    "```\n\n"
    "``final_decision`` MUST be one of: ``BUY YES``, ``BUY NO``, or ``PASS``.\n"
    "For ``PASS``, set ``bet_amount_usd`` to 0 and ``token_id`` to null.\n\n"
    "Output ONLY the JSON — no extra commentary."
)

PERSONAS = [
    ("Base Neutral", "Analyze the market objectively using your tools and make the most logical decision."),
    ("Contrarian", "You are a contrarian trader. The market is often wrong and overreacts to hype. Use your tools to find evidence that the current favorite outcome will fail. Look for hidden risks."),
    ("Momentum Trader", "You are a momentum trader. You believe the crowd is usually right. Use your tools to find breaking news or social media sentiment that confirms the current price trend is going to accelerate."),
    ("Quantitative Arbitrager", "You are highly risk-averse. Focus heavily on the get_market_liquidity_and_fees and calculate_expected_value tools. Only take a position if the spread is tight and the mathematical expected value is exceptionally high. Otherwise, PASS."),
    ("Social Sentiment Scraper", "You ignore traditional news. You believe markets are driven purely by social media mechanics. Rely heavily on the search_x_twitter tool to gauge retail emotion before making a trade."),
]

print(f'Loaded {len(PERSONAS)} personas')

In [ ]:
# Cell 3b: Tool definitions (OpenAI format)
TOOLS = [
    {"type": "function", "function": {"name": "get_order_book", "description": "Get the current order book for a Polymarket market.", "parameters": {"properties": {"condition_id": {"title": "Condition Id", "type": "string"}}, "required": ["condition_id"], "title": "get_order_bookArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "get_price_history", "description": "Get the resolved price history for a market.", "parameters": {"properties": {"condition_id": {"title": "Condition Id", "type": "string"}}, "required": ["condition_id"], "title": "get_price_historyArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "read_webpage", "description": "Read the full text content of a webpage.", "parameters": {"properties": {"url": {"title": "Url", "type": "string"}}, "required": ["url"], "title": "read_webpageArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "websearch", "description": "Search the web using DuckDuckGo.", "parameters": {"properties": {"query": {"title": "Query", "type": "string"}, "time_filter": {"default": None, "title": "Time Filter", "type": "string"}}, "required": ["query"], "title": "websearchArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "search_active_markets", "description": "Search Polymarket markets matching a keyword.", "parameters": {"properties": {"query": {"title": "Query", "type": "string"}}, "required": ["query"], "title": "search_active_marketsArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "get_market_liquidity_and_fees", "description": "Get liquidity depth and fee schedule for a Polymarket market.", "parameters": {"properties": {"condition_id": {"title": "Condition Id", "type": "string"}}, "required": ["condition_id"], "title": "get_market_liquidity_and_feesArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "search_x_twitter", "description": "Search X (formerly Twitter) for recent posts about a topic.", "parameters": {"properties": {"query": {"title": "Query", "type": "string"}}, "required": ["query"], "title": "search_x_twitterArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "calculate_expected_value", "description": "Calculate the expected value of a bet.", "parameters": {"properties": {"probability": {"title": "Probability", "type": "number"}, "payout": {"title": "Payout", "type": "number"}, "cost": {"title": "Cost", "type": "number"}}, "required": ["probability", "payout", "cost"], "title": "calculate_expected_valueArguments", "type": "object"}}},
    {"type": "function", "function": {"name": "calculator", "description": "A simple calculator that can evaluate mathematical expressions.", "parameters": {"properties": {"expression": {"title": "Expression", "type": "string"}}, "required": ["expression"], "title": "calculatorArguments", "type": "object"}}},
]
print(f'Loaded {len(TOOLS)} tool definitions')

In [ ]:
# Cell 3c: Tool implementations
CLOB_HOST = 'https://clob.polymarket.com'
USER_AGENT = 'polybrrr-rl/0.1'
REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_SLEEP = 1.0

def _get_with_retry(url, params=None):
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, params=params, headers={'User-Agent': USER_AGENT}, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            last_err = exc
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP * attempt)
    return {'error': str(last_err)}

def _clob_market_tokens(condition_id):
    try:
        resp = requests.get(f'{CLOB_HOST}/clob-markets/{condition_id}', headers={'User-Agent': USER_AGENT}, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        return resp.json().get('t')
    except Exception:
        return None

def get_order_book(condition_id):
    try:
        tokens = _clob_market_tokens(condition_id)
        if not tokens or not tokens[0]:
            return json.dumps({'error': f'Could not resolve CLOB market info for {condition_id}'}, indent=2)
        entries = []
        for tok in tokens:
            tid = tok.get('t') or tok.get('token_id')
            if not tid: continue
            mp = _get_with_retry(f'{CLOB_HOST}/midpoint', params={'token_id': tid})
            sp = _get_with_retry(f'{CLOB_HOST}/spread', params={'token_id': tid})
            lp = _get_with_retry(f'{CLOB_HOST}/last-trade-price', params={'token_id': tid})
            book = _get_with_retry(f'{CLOB_HOST}/book', params={'token_id': tid})
            bids = book.get('bids', []) or []
            asks = book.get('asks', []) or []
            midpoint = float(mp.get('mid')) if 'mid' in mp else None
            def _top_near(levels, ref_price, n=5):
                if not levels or ref_price is None: return []
                sorted_levels = sorted(levels, key=lambda l: abs(float(l['price']) - ref_price))
                return [{'price': lvl['price'], 'size': lvl['size']} for lvl in sorted_levels[:n]]
            entries.append({'token_id': tid, 'midpoint': midpoint, 'spread': float(sp.get('spread')) if 'spread' in sp else None, 'last_trade_price': float(lp.get('price')) if 'price' in lp else None, 'last_trade_side': lp.get('side') if 'side' in lp else None, 'bid_depth_total': sum(float(b['size']) for b in bids), 'ask_depth_total': sum(float(a['size']) for a in asks), 'levels_near_mid': {'bids': _top_near(bids, midpoint), 'asks': _top_near(asks, midpoint)}})
        return json.dumps({'condition_id': condition_id, 'tokens': entries}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def get_price_history(condition_id):
    return json.dumps({'error': 'Price history not available in Colab'}, indent=2)

def read_webpage(url):
    try:
        jina_url = f'https://r.jina.ai/{url}'
        resp = requests.get(jina_url, headers={'User-Agent': USER_AGENT, 'Accept': 'text/markdown'}, timeout=15)
        resp.raise_for_status()
        content = resp.text
        max_chars = 16000
        if len(content) > max_chars:
            content = content[:max_chars] + '\n\n...[CONTENT TRUNCATED]...'
        return json.dumps({'url': url, 'content': content, 'truncated': len(resp.text) > max_chars}, indent=2)
    except Exception as exc:
        return json.dumps({'error': f'Failed to read webpage: {exc}'}, indent=2)

def websearch(query, time_filter=None):
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(query, timelimit=time_filter, max_results=8))
            results = [{'title': r['title'], 'url': r['href'], 'snippet': r.get('body', '')} for r in raw]
        return json.dumps({'query': query, 'time_filter': time_filter, 'results': results}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def search_active_markets(query):
    try:
        resp = requests.get(f'{CLOB_HOST}/sampling-markets', params={'limit': 200}, headers={'User-Agent': USER_AGENT}, timeout=REQUEST_TIMEOUT)
        matches = []
        if resp.status_code == 200:
            keywords = [kw for kw in query.lower().split() if len(kw) >= 3]
            for m in (resp.json().get('data') or []):
                question = (m.get('question') or '').lower()
                matched = [kw for kw in keywords if kw in question]
                ratio = len(matched) / len(keywords) if keywords else 0
                if ratio >= 0.6 or len(matched) >= 2:
                    tokens = m.get('tokens', []) or []
                    matches.append({'condition_id': m.get('condition_id'), 'question': m.get('question'), 'market_slug': m.get('market_slug'), 'closed': m.get('closed'), 'active': m.get('active'), 'match_score': round(ratio, 2), 'tokens': [{'token_id': t.get('token_id'), 'outcome': t.get('outcome')} for t in tokens]})
        return json.dumps({'query': query, 'matches': matches[:50], 'count': len(matches)}, indent=2, ensure_ascii=False)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def get_market_liquidity_and_fees(condition_id):
    try:
        clob_resp = requests.get(f'{CLOB_HOST}/clob-markets/{condition_id}', headers={'User-Agent': USER_AGENT}, timeout=REQUEST_TIMEOUT)
        clob_data = clob_resp.json() if clob_resp.status_code == 200 else {}
        tokens_arr = clob_data.get('t') or []
        if not tokens_arr:
            return json.dumps({'error': f'Could not resolve token for {condition_id}'}, indent=2)
        market_data = _get_with_retry(f'{CLOB_HOST}/markets/{condition_id}')
        if 'error' in market_data:
            return json.dumps(market_data, indent=2)
        token_outcomes = market_data.get('tokens', []) or []
        book_entries = []
        for i, token_info in enumerate(tokens_arr):
            tid = token_info.get('t') or token_info.get('token_id')
            if not tid: continue
            outcome = token_outcomes[i].get('outcome') if i < len(token_outcomes) else f'Token {i}'
            mp = _get_with_retry(f'{CLOB_HOST}/midpoint', params={'token_id': tid})
            sp = _get_with_retry(f'{CLOB_HOST}/spread', params={'token_id': tid})
            lp = _get_with_retry(f'{CLOB_HOST}/last-trade-price', params={'token_id': tid})
            bk = _get_with_retry(f'{CLOB_HOST}/book', params={'token_id': tid})
            bids = bk.get('bids', []) or []
            asks = bk.get('asks', []) or []
            midpoint = float(mp.get('mid')) if 'mid' in mp else None
            def _depth_near(levels, ref_price, threshold):
                if not levels or ref_price is None: return 0.0
                total = 0.0
                for lvl in levels:
                    p = float(lvl['price'])
                    if abs(p - ref_price) / ref_price <= threshold: total += float(lvl['size'])
                return total
            def _top_near(levels, ref_price, n=5):
                if not levels or ref_price is None: return []
                sorted_levels = sorted(levels, key=lambda l: abs(float(l['price']) - ref_price))
                return [{'price': lvl['price'], 'size': lvl['size']} for lvl in sorted_levels[:n]]
            book_entries.append({'token_id': tid, 'outcome': outcome, 'price': midpoint, 'spread': float(sp.get('spread')) if 'spread' in sp else None, 'last_trade_price': float(lp.get('price')) if 'price' in lp else None, 'last_trade_side': lp.get('side') if 'side' in lp else None, 'bid_depth_total': sum(float(b['size']) for b in bids), 'ask_depth_total': sum(float(a['size']) for a in asks), 'bid_depth_near_mid': _depth_near(bids, midpoint, 0.20) if midpoint else None, 'ask_depth_near_mid': _depth_near(asks, midpoint, 0.20) if midpoint else None, 'levels_near_mid': {'bids': _top_near(bids, midpoint), 'asks': _top_near(asks, midpoint)}})
        token_count = len(token_outcomes)
        neg_risk = clob_data.get('nr', False) or market_data.get('neg_risk') or False
        fd = clob_data.get('fd') or {}
        if fd.get('r'): taker_fee_bps = int(float(fd['r']) * 10000)
        elif neg_risk or token_count <= 2: taker_fee_bps = 200
        elif token_count <= 5: taker_fee_bps = 300
        elif token_count <= 10: taker_fee_bps = 400
        else: taker_fee_bps = 500
        return json.dumps({'condition_id': condition_id, 'question': market_data.get('question'), 'token_count': token_count, 'neg_risk': neg_risk, 'taker_fee_percent': round(taker_fee_bps / 100, 2), 'taker_fee_bps': taker_fee_bps, 'tokens': book_entries, 'minimum_tick_size': clob_data.get('mts') or market_data.get('minimum_tick_size'), 'minimum_order_size': market_data.get('minimum_order_size')}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def search_x_twitter(query):
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            raw = list(ddgs.text(f'site:x.com {query}', max_results=10))
            results = [{'title': r['title'], 'url': r['href'], 'snippet': r.get('body', '')} for r in raw if 'x.com' in r['href'].lower() or 'twitter.com' in r['href'].lower()]
        return json.dumps({'query': query, 'results': results, 'count': len(results)}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def calculate_expected_value(probability, payout, cost):
    try:
        probability, payout, cost = float(probability), float(payout), float(cost)
        ev = (probability * payout) - cost
        roi = (ev / cost * 100) if cost > 0 else 0.0
        return json.dumps({'probability': probability, 'payout': payout, 'cost': cost, 'expected_value': round(ev, 6), 'roi_percent': round(roi, 2), 'profitable': ev > 0}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def calculator(expression):
    try:
        allowed = re.compile(r'^[\d\s+\-*/().,%eE]+$')
        if not allowed.match(expression.replace(' ', '')):
            return json.dumps({'error': 'Expression contains disallowed characters'}, indent=2)
        result = eval(expression, {'__builtins__': {}}, {})
        return json.dumps({'expression': expression, 'result': result}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

def execute_tool(name, args):
    try:
        if name == 'get_order_book': return get_order_book(args['condition_id'])
        elif name == 'get_market_liquidity_and_fees': return get_market_liquidity_and_fees(args['condition_id'])
        elif name == 'websearch': return websearch(args['query'], args.get('time_filter'))
        elif name == 'read_webpage': return read_webpage(args['url'])
        elif name == 'search_x_twitter': return search_x_twitter(args['query'])
        elif name == 'get_price_history': return get_price_history(args['condition_id'])
        elif name == 'search_active_markets': return search_active_markets(args['query'])
        elif name == 'calculate_expected_value': return calculate_expected_value(float(args['probability']), float(args['payout']), float(args['cost']))
        elif name == 'calculator': return calculator(args['expression'])
        else: return json.dumps({'error': f'Unknown tool: {name}'})
    except Exception as exc:
        return json.dumps({'error': str(exc)})

print('Tool implementations loaded')

In [ ]:
# Cell 3d: Parsing and helper functions
THINK_START = '<think>'
THINK_END = '</think>'

# NOTE: BS/NL dodge a transport that strips backslashes; patterns equal the plain regexes.
# Qwen3 server format: <tool_call> blocks AFTER </think> (old pattern wrongly required them inside <think>).
BS = chr(92)
NL = chr(10)
TOOL_CALL_RE = re.compile('<tool_call>' + BS + 's*' + NL + '<function=(' + BS + 'w+)>(.*?)</function>' + BS + 's*' + NL + '</tool_call>', re.DOTALL)
PARAM_RE = re.compile('<parameter=(' + BS + 'w+)>' + NL + '(.*?)' + NL + '</parameter>', re.DOTALL)


def parse_generated_response(generated_text):
    reasoning_content = ''
    content = generated_text
    if THINK_START in generated_text:
        parts = generated_text.split(THINK_START, 1)
        if len(parts) == 2:
            after = parts[1]
            if THINK_END in after:
                think_parts = after.split(THINK_END, 1)
                reasoning_content = think_parts[0].strip()
                content = think_parts[1] if len(think_parts) == 2 else ''
            else:
                reasoning_content = after.strip()
                content = ''
    elif THINK_END in generated_text:
        think_parts = generated_text.split(THINK_END, 1)
        reasoning_content = think_parts[0].strip()
        content = think_parts[1] if len(think_parts) == 2 else ''
    content = content.strip()
    tool_calls = []
    for match in TOOL_CALL_RE.finditer(content):
        fn_name = match.group(1)
        fn_body = match.group(2)
        arguments = {}
        for pm in PARAM_RE.finditer(fn_body):
            arg_name = pm.group(1)
            arg_value = pm.group(2).strip()
            try:
                arguments[arg_name] = json.loads(arg_value)
            except (json.JSONDecodeError, ValueError):
                arguments[arg_name] = arg_value
        tool_calls.append({'type': 'function', 'function': {'name': fn_name, 'arguments': arguments}})
    if tool_calls:
        content = TOOL_CALL_RE.sub('', content).strip()
    return reasoning_content, content, tool_calls


def normalize_messages(messages):
    normalized = []
    for message in messages:
        message = dict(message)
        if message.get('content') is None:
            message['content'] = ''
        reasoning = message.get('reasoning')
        if reasoning and 'reasoning_content' not in message:
            message['reasoning_content'] = reasoning
        tool_calls = message.get('tool_calls')
        if tool_calls:
            normalized_tcs = []
            for tc in tool_calls:
                tc = dict(tc)
                fn = tc.get('function')
                if fn:
                    fn = dict(fn)
                    args = fn.get('arguments')
                    if isinstance(args, str):
                        try:
                            fn['arguments'] = json.loads(args)
                        except json.JSONDecodeError:
                            pass
                    tc['function'] = fn
                normalized_tcs.append(tc)
            message['tool_calls'] = normalized_tcs
        normalized.append(message)
    return normalized


def mask_assistant_tokens(tokenizer, messages):
    messages = normalize_messages(messages)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    enc = tokenizer(full_text, return_offsets_mapping=True, return_tensors='pt')
    full_ids = enc['input_ids'][0]
    offsets = enc['offset_mapping'][0]
    labels = full_ids.clone()
    labels[:] = -100
    IM_START = '<|im_start|>assistant' + NL
    IM_END = '<|im_end|>' + NL
    pos = 0
    while True:
        a_start = full_text.find(IM_START, pos)
        if a_start == -1:
            break
        content_start = a_start + len(IM_START)
        content_end = full_text.find(IM_END, content_start)
        if content_end == -1:
            content_end = len(full_text)
        else:
            content_end += len(IM_END)
        tok_start = tok_end = None
        for i, (cs, ce) in enumerate(offsets):
            cs, ce = cs.item(), ce.item()
            if tok_start is None and ce > content_start:
                tok_start = i
            if cs >= content_end:
                tok_end = i
                break
        if tok_start is not None:
            if tok_end is None:
                tok_end = len(full_ids)
            labels[tok_start:tok_end] = full_ids[tok_start:tok_end]
        pos = content_end
    return full_ids, labels


def get_resolution_rules(condition_id):
    try:
        data = _get_with_retry(f'{CLOB_HOST}/markets/{condition_id}')
        if 'error' in data:
            return json.dumps({'error': data['error']}, indent=2)
        tokens = data.get('tokens', []) or []
        return json.dumps({'condition_id': condition_id, 'question': data.get('question'), 'description': data.get('description'), 'market_slug': data.get('market_slug'), 'end_date': data.get('end_date_iso'), 'closed': data.get('closed'), 'outcomes': [{'token_id': t.get('token_id'), 'outcome': t.get('outcome')} for t in tokens]}, indent=2)
    except Exception as exc:
        return json.dumps({'error': str(exc)}, indent=2)

print('Parsing and helper functions loaded')

In [ ]:
# Cell 4: Load market data (FULL pool) and build (system_prompt, market_context) pairs
# Split policy: 900/20/0 repartition handled by the training-loop cell, which takes
# the last 20 unique markets as held-out eval and the previous 900 as the train pool
# (any front remainder is discarded). This cell therefore passes ALL markets through
# as TRAIN_MARKETS; val/test stay empty for API compatibility.
import random
if MARKETS_URL and not Path(MARKETS_JSON).exists():
    resp = requests.get(MARKETS_URL)
    Path(MARKETS_JSON).write_text(resp.text)

with open(MARKETS_JSON) as f:
    snapshot = json.load(f)

markets = snapshot.get('markets', [])
print(f'Loaded {len(markets)} markets from {MARKETS_JSON}')

TRAIN_MARKETS = list(markets)
VAL_MARKETS = []
TEST_MARKETS = []
print(f'Split: train={len(TRAIN_MARKETS)} val=0 test=0 markets (full pool; 900/20 eval+rotation split happens in the training-loop cell)')

def _pairs_for(market_list):
    pairs = []
    for market in market_list:
        condition_id = market['condition_id']
        question = market.get('question', '')
        end_date = market.get('end_date_iso', '')
        rules_json = get_resolution_rules(condition_id)
        rules = json.loads(rules_json) if isinstance(rules_json, str) else rules_json
        description = rules.get('description', '') if isinstance(rules, dict) else ''
        outcomes = json.dumps(rules.get('outcomes', [])) if isinstance(rules, dict) else '[]'
        today_str = datetime.now(UTC).strftime('%Y-%m-%d')
        market_context = f'''
MARKET TO EVALUATE
==================
Today: {today_str}
Condition ID: {condition_id}
Question: {question}
Description: {description}
End Date: {end_date}
Outcomes: {outcomes}

Your confidence_score controls what fraction of the visible order-book depth
you deploy (0-100).  Higher confidence = larger position.

Research this market using the available tools before deciding.
'''
        for persona_name, persona_prompt in PERSONAS:
            system_prompt = BASE_PROMPT + '\n\n' + persona_prompt
            pairs.append((system_prompt, market_context, persona_name, condition_id))
    return pairs

task_pairs = _pairs_for(TRAIN_MARKETS)
val_pairs = _pairs_for(VAL_MARKETS)
test_pairs = _pairs_for(TEST_MARKETS)
print(f'Built {len(task_pairs)} train + {len(val_pairs)} val + {len(test_pairs)} test pairs ({len(PERSONAS)} personas each)')


In [ ]:
'Step 4: start vLLM engine (also defines stop_server, used in step 6)'
import subprocess, time, urllib.request, gc

SERVER_PORT = 8000
SERVER_LOG = '/tmp/vllm_server.log'
ROLLOUT_MODEL = VLLM_MODEL
_SERVED_ADAPTER = 'none'

def _port_busy():
    import socket
    s = socket.socket()
    try:
        s.bind(('127.0.0.1', SERVER_PORT))
        return False
    except OSError:
        return True
    finally:
        s.close()

def _gpu_used_mb():
    out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
    try:
        return int(out.split()[0])
    except Exception:
        return -1

def _wait_ready(proc, timeout_s=1500):
    for i in range(int(timeout_s / 10)):
        time.sleep(10)
        if proc.poll() is not None:
            print('SERVER EXITED, code', proc.returncode)
            print(subprocess.run(['tail', '-15', SERVER_LOG], capture_output=True, text=True).stdout[-2000:])
            return False
        try:
            r = urllib.request.urlopen('http://127.0.0.1:' + str(SERVER_PORT) + '/health', timeout=5)
            if r.status == 200:
                print('server READY after ' + str((i + 1) * 10) + 's')
                return True
        except Exception:
            pass
        if (i + 1) % 6 == 0:
            print('  still loading... ' + str((i + 1) * 10) + 's elapsed')
    return False

def stop_server():
    subprocess.run('pkill -f vllm', shell=True)
    time.sleep(5)
    for i in range(24):
        if not _port_busy():
            break
        time.sleep(10)
    used = _gpu_used_mb()
    if used > 2000:
        print('port free but GPU still holds ' + str(used) + 'MB - SIGKILL leftovers')
        subprocess.run('pkill -9 -f vllm', shell=True)
        time.sleep(10)
    try:
        import torch
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception as e:
        print('cuda free note:', e)
    time.sleep(10)
    used = _gpu_used_mb()
    print('GPU used after stop: ' + str(used) + 'MB')
    if used > 2000:
        print('WARNING: GPU not released - do not start training, inspect processes first')
        return False
    return True

def start_server(adapter_path=None):
    global ROLLOUT_MODEL, _SERVED_ADAPTER
    stop_server()
    gc.collect()
    cmd = [
        'vllm', 'serve', VLLM_MODEL,
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '--trust-remote-code',
        '--enable-lora', '--max-lora-rank', '64',
        '--max-model-len', str(MAX_COMPLETION_LENGTH),
        '--gpu-memory-utilization', '0.90',
        '--kv-cache-dtype', 'fp8',
        '--enable-prefix-caching',
        '--max-num-seqs', str(MAX_CONCURRENT_GENERATION_AGENTS),
        '--max-num-batched-tokens', '8192',
        '--language-model-only'
    ]
    if adapter_path:
        cmd += ['--lora-modules', 'policy_adapter=' + adapter_path]
        ROLLOUT_MODEL = 'policy_adapter'
    else:
        ROLLOUT_MODEL = VLLM_MODEL
    logf = open(SERVER_LOG, 'w')
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    print('server pid:', proc.pid, 'adapter:', adapter_path)
    ok = _wait_ready(proc)
    if ok:
        _SERVED_ADAPTER = adapter_path or 'none'
    print('ROLLOUT_MODEL =', ROLLOUT_MODEL, 'ready =', ok)
    return ok

def ensure_server(adapter_path=None):
    global _SERVED_ADAPTER
    want = adapter_path or 'none'
    if _SERVED_ADAPTER == want:
        try:
            r = urllib.request.urlopen('http://127.0.0.1:' + str(SERVER_PORT) + '/health', timeout=5)
            if r.status == 200:
                print('server already up with adapter:', want)
                return True
        except Exception:
            pass
    return start_server(adapter_path)

# stop_server()
# ensure_server(LORA_ADAPTER_PATH if 'LORA_ADAPTER_PATH' in dir() else None)

In [ ]:
'Step 5a: rollout client (async, 8 concurrent generations, tools run alongside)'
import asyncio, time, json
from openai import AsyncOpenAI
import concurrent.futures as cfutures

if 'tok_local' not in dir():
    from transformers import AutoTokenizer
    tok_local = AutoTokenizer.from_pretrained(TRAINING_MODEL, trust_remote_code=True)
    print('local CPU tokenizer ready')

if 'ROLLOUT_MODEL' not in dir():
    ROLLOUT_MODEL = VLLM_MODEL
ACL = AsyncOpenAI(base_url='http://127.0.0.1:8000/v1', api_key='x', max_retries=0)
GEN_SEM = asyncio.Semaphore(MAX_CONCURRENT_GENERATION_AGENTS)
_tool_poolA = cfutures.ThreadPoolExecutor(max_workers=64)

def _render(msgs, tools, add_gen=True):
    return tok_local.apply_chat_template(msgs, tools=tools, add_generation_prompt=add_gen, tokenize=False)

async def _complete(prompt, tries=3):
    last = None
    for a in range(tries):
        try:
            async with GEN_SEM:
                r = await ACL.completions.create(model=ROLLOUT_MODEL, prompt=prompt, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, timeout=600)
            c = r.choices[0]
            return c.text, (r.usage.completion_tokens if r.usage else 0)
        except Exception as e:
            last = e
            print('    request attempt ' + str(a + 1) + ' failed: ' + type(e).__name__ + '; retrying...')
            await asyncio.sleep(5)
    raise last

async def _run_agent_async(sys_prompt, mkt_ctx, persona_name, condition_id, max_rounds):
    t0 = time.time()
    messages = [{'role': 'system', 'content': sys_prompt}, {'role': 'user', 'content': mkt_ctx}]
    total_tokens = 0
    turns = 0
    for rnd in range(max_rounds):
        force_final = rnd >= max_rounds - 1
        if force_final:
            msgs = messages + [{'role': 'user', 'content': 'STOP calling tools now. Produce your final JSON IMMEDIATELY.'}]
            prompt = _render(msgs, None)
        else:
            prompt = _render(messages, TOOLS)
        text, ntok = await _complete(prompt)
        total_tokens += ntok
        turns += 1
        reasoning_content, content, tool_calls = parse_generated_response(text)
        asst = {'role': 'assistant', 'content': content}
        if reasoning_content:
            asst['reasoning_content'] = reasoning_content
        if tool_calls:
            asst['tool_calls'] = tool_calls
        messages.append(asst)
        if tool_calls and not force_final:
            loop = asyncio.get_running_loop()
            futs = [loop.run_in_executor(_tool_poolA, execute_tool, tc['function']['name'], tc['function']['arguments']) for tc in tool_calls]
            for res in await asyncio.gather(*futs):
                messages.append({'role': 'tool', 'content': res})
        else:
            break
    return {'condition_id': condition_id, 'persona': persona_name, 'system_prompt': sys_prompt, 'market_context': mkt_ctx, 'messages': messages, 'old_logprobs': None, 'token_ids_per_turn': [], '_gen_tokens': total_tokens, '_turns': turns, '_wall': time.time() - t0}

async def generate_traces_async(task_pairs_batch, max_rounds=None):
    """
    Spawn N workers, each in charge of 1 agent generating its trace at a time.
    Having workers instead of just sending task_pairs_batch agents help not overloading vLLM KV cache so it can use prefix caching.
    Combining with asyncio.Semaphore in _complete(), it allows to only have the gpu generating for MAX_CONCURRENT_GENERATION_AGENTS at a time while the remaining one are executing tools.
    """
    mr = max_rounds if max_rounds is not None else MAX_TOOL_ROUNDS
    t0 = time.time()
    total_tasks = len(task_pairs_batch)
    print('  Async rollout: ' + str(total_tasks) + ' agents, model=' + ROLLOUT_MODEL)

    # Put all tasks into an async queue
    queue = asyncio.Queue()
    for tp in task_pairs_batch:
        queue.put_nowait(tp)

    traces = []
    lock = asyncio.Lock()
    done = 0

    # Define a worker that constantly pulls tasks until the queue is empty
    async def worker():
        nonlocal done
        while True:
            try:
                tp = queue.get_nowait()
            except asyncio.QueueEmpty:
                break

            try:
                # Run the agent (this takes multiple turns)
                tr = await _run_agent_async(tp[0], tp[1], tp[2], tp[3], mr)
            except Exception as exc:
                async with lock:
                    done += 1
                    print('    [' + str(done) + '/' + str(total_tasks) + '] ERROR: ' + type(exc).__name__ + ': ' + str(exc)[:300])
            else:
                async with lock:
                    done += 1
                    traces.append(tr)
                    print('    [' + str(done) + '/' + str(total_tasks) + '] ' + tr['persona'] + ': ' + str(tr['_turns']) + ' turns, ' + str(tr['_gen_tokens']) + ' tok')
            finally:
                queue.task_done()

    workers = [asyncio.create_task(worker()) for _ in range(NB_WORKERS_GENERATION)]

    await asyncio.gather(*workers)

    dt = time.time() - t0
    tot = sum(t['_gen_tokens'] for t in traces)
    print('  Async done: ' + str(len(traces)) + ' traces in ' + str(round(dt, 1)) + 's')

    try:
        import wandb
        if wandb.run is not None:
            wandb.log({'traces/async_wall_sec': dt, 'traces/async_tok_per_sec': tot / dt if dt > 0 else 0})
    except Exception:
        pass

    return list(traces)
print('rollout client ready, model=' + ROLLOUT_MODEL)

In [ ]:
'Step 5b: scoring - old logprobs with adapter, ref probs with base model'
from openai import OpenAI
import concurrent.futures as cfutures2, math

SCL = OpenAI(base_url='http://127.0.0.1:8000/v1', api_key='x')
_score_pool = cfutures2.ThreadPoolExecutor(max_workers=MAX_CONCURRENT_GENERATION_AGENTS)

def _score_trace(trace, model):
    full_text = tok_local.apply_chat_template(trace['messages'], tokenize=False)
    full_ids, labels = mask_assistant_tokens(tok_local, trace['messages'])
    ids = full_ids.tolist()
    labs = labels.tolist()
    r = SCL.completions.create(model=model, prompt=full_text, max_tokens=1, temperature=0, echo=True, logprobs=5)
    ch = r.choices[0]
    toks = ch.logprobs.tokens
    tlps = ch.logprobs.token_logprobs
    tops = ch.logprobs.top_logprobs
    n_prompt = len(ids)
    assert len(toks) == n_prompt + 1, 'token mismatch'
    old_lp, ref_p, nfb = [], [], 0
    for i in range(n_prompt):
        if labs[i] == -100 or i == 0:
            continue
        lp = tlps[i]
        if lp is None:
            d = tops[i] or {}
            lp = next(iter(d.values())) if d else 0.0
            nfb += 1
        old_lp.append({'token_id': ids[i], 'log_prob': lp})
        ref_p.append({'token_id': ids[i], 'prob': math.exp(lp) if lp > -50 else 0.0})
    return old_lp, ref_p, nfb

def _score_all(traces, model, field):
    res = list(_score_pool.map(lambda tr: _score_trace(tr, model), traces))
    for tr, (old_lp, ref_p, nfb) in zip(traces, res):
        if field == 'old':
            tr['old_logprobs'] = old_lp
        else:
            tr['ref_probs'] = ref_p
    nfb_tot = sum(r[2] for r in res)
    ntok = sum(len(r[0]) for r in res)
    print('  scored ' + str(len(traces)) + ' traces (' + field + '): ' + str(ntok) + ' tokens, ' + str(nfb_tot) + ' fallbacks')

def compute_old_logprobs(traces, model=None):
    _score_all(traces, model or ROLLOUT_MODEL, 'old')

def compute_ref_probs(traces):
    _score_all(traces, VLLM_MODEL, 'ref')

print('scoring ready: compute_old_logprobs(traces), compute_ref_probs(traces)')

In [ ]:
# Cell 5b-ii: TRL fix for offline rollout + custom rewards
# Patches TRL's grpo_trainer.py so reward functions also receive 'completions'.
# Run ONCE PER VM, BEFORE the first cell that imports trl (the loop cells import
# GRPOTrainer at call time). If trl was already imported, the file patch has no
# effect on the loaded module -> restart the runtime and re-run the setup cells.
import sys
import importlib.util
import pathlib

spec = importlib.util.find_spec('trl')   # locates trl WITHOUT importing it
assert spec is not None and spec.origin, (
    'trl is not installed in this VM yet - run Cell 1 (pip installs) first, then re-run '
    'this cell BEFORE any cell that imports trl (the loop cells).')
p = pathlib.Path(spec.origin).parent / 'trainer' / 'grpo_trainer.py'
assert p.exists(), 'unexpected TRL layout, cannot find: ' + str(p)

s = p.read_text()
old = '["prompt", "completion", "completion_ids"]'
new = '["prompt", "completion", "completions", "completion_ids"]'
if new in s and old not in s:
    print('TRL reward-kwargs patch already present (file is patched)')
elif old in s:
    p.write_text(s.replace(old, new))
    print('TRL reward-kwargs patch applied to ' + str(p))
else:
    raise RuntimeError(
        'PATCH ANCHOR NOT FOUND - TRL changed the reward-kwargs list upstream '
        '(version drift). The offline rollout will not pass completions to the '
        'reward function. Inspect reward_kwargs in ' + str(p) + ' and update this patch.')

if 'trl' in sys.modules:
    print('WARNING: trl is already imported in this kernel. The file patch does NOT '
          'affect the loaded module. Restart the runtime (or move this cell before the '
          'first trl import) or the reward function will not receive completions.')

In [ ]:
# Cell 5c: Reward functions (search_x_twitter usage) + debugging
# Defines the two missing helpers used by _rl_loop and _rl_overfit:
#   evaluate_reward(traces, label) -> (mean, rate)
#   search_x_twitter_reward_func(prompts, completions, ...) -> list[float] for GRPOTrainer
#
# Spec (from notebook header): +1 if the 'search_x_twitter' tool was called, -1 otherwise.

def _messages_used_search_x_twitter(messages):
    """Return True if any assistant message called the 'search_x_twitter' tool."""
    if messages is None:
        return False
    # Raw string fallback (TRL sometimes passes decoded text).
    if isinstance(messages, str):
        return 'search_x_twitter' in messages or '<function=search_x_twitter>' in messages
    if not isinstance(messages, list):
        try:
            messages = list(messages)
        except Exception:
            return False
    for m in messages:
        if not isinstance(m, dict):
            continue
        # Standard path: assistant tool_calls list.
        for tc in (m.get('tool_calls') or []):
            try:
                if not isinstance(tc, dict):
                    continue
                fn = tc.get('function') or {}
                if isinstance(fn, dict) and fn.get('name') == 'search_x_twitter':
                    return True
            except Exception:
                continue
        # Fallback: raw <tool_call> XML still present in content.
        try:
            content = m.get('content') or ''
            if isinstance(content, str) and '<function=search_x_twitter>' in content:
                return True
        except Exception:
            pass
    return False

def _reward_for_messages(messages):
    return 1.0 if _messages_used_search_x_twitter(messages) else -1.0

def evaluate_reward(traces, label=''):
    """Score a list of rollout traces. Returns (mean_reward, calc_rate_pct)."""
    if not traces:
        print(f'[{label}] WARNING: empty traces list, returning (0.0, 0.0)')
        return 0.0, 0.0
    rewards = []
    tool_hist = {}
    per_trace = []
    for i, tr in enumerate(traces):
        msgs = tr.get('messages', []) if isinstance(tr, dict) else tr
        r = _reward_for_messages(msgs)
        rewards.append(r)
        # Collect tool usage for debugging.
        try:
            seen = set()
            for m in (msgs if isinstance(msgs, list) else []):
                if isinstance(m, dict):
                    for tc in (m.get('tool_calls') or []):
                        try:
                            n = (tc.get('function') or {}).get('name', '?')
                            seen.add(n)
                            tool_hist[n] = tool_hist.get(n, 0) + 1
                        except Exception:
                            pass
            per_trace.append((i, r, sorted(seen),
                              tr.get('persona', '?') if isinstance(tr, dict) else '?'))
        except Exception as e:
            per_trace.append((i, r, [f'parse-error: {e}'], '?'))
    n = len(rewards)
    n_pos = sum(1 for r in rewards if r > 0)
    n_neg = n - n_pos
    mean = sum(rewards) / n if n else 0.0
    rate = 100.0 * n_pos / n if n else 0.0
    print(f'[{label}] n={n} mean={mean:.4f} calc_rate={rate:.1f}% (+1:{n_pos} -1:{n_neg})')
    if tool_hist:
        top = sorted(tool_hist.items(), key=lambda kv: -kv[1])[:10]
        print(f'[{label}] tool_calls total={sum(tool_hist.values())} top={top}')
    else:
        print(f'[{label}] WARNING: no tool_calls found in any trace (all rewards will be -1)')
    # Show a few examples of each class for quick inspection.
    pos_ex = [t for t in per_trace if t[1] > 0][:3]
    neg_ex = [t for t in per_trace if t[1] <= 0][:3]
    for idx, r, tools, persona in pos_ex:
        print(f'[{label}]  +1 ex trace[{idx}] persona={persona} tools={tools}')
    for idx, r, tools, persona in neg_ex:
        print(f'[{label}]  -1 ex trace[{idx}] persona={persona} tools={tools}')
    # Persona breakdown.
    try:
        from collections import defaultdict
        by_persona = defaultdict(list)
        for idx, r, tools, persona in per_trace:
            by_persona[persona].append(r)
        for persona, rs in sorted(by_persona.items()):
            m = sum(rs) / len(rs)
            print(f'[{label}]  persona {persona}: n={len(rs)} mean={m:.3f}')
    except Exception as e:
        print(f'[{label}] persona breakdown skipped: {e}')
    return mean, rate

def search_x_twitter_reward_func(*args, **kwargs):
    """TRL GRPOTrainer reward function. Returns one float per completion."""
    # Accept all TRL calling styles: (prompts, completions), or keywords including completion_ids.
    if len(args) >= 2:
        prompts, completions = args[0], args[1]
    else:
        prompts = kwargs.pop('prompts', args[0] if len(args) == 1 else [])
        completions = kwargs.pop('completions', [])
    kwargs.pop('completion_ids', None)
    rewards = []
    for comp in completions:
        try:
            rewards.append(_reward_for_messages(comp))
        except Exception as e:
            print(f'[reward_func] WARNING: failed to score completion ({type(e).__name__}: {e}), assigning -1')
            rewards.append(-1.0)
    # Light debugging for the first call of each training run.
    if not hasattr(search_x_twitter_reward_func, '_logged_once'):
        try:
            import collections
            c = collections.Counter(rewards)
            print(f'[reward_func] first call: n={len(rewards)} dist={dict(c)} comp_type={type(completions[0]).__name__ if completions else None}')
            search_x_twitter_reward_func._logged_once = True
        except Exception:
            pass
    return rewards

# --- Self-test (no GPU, no rollout needed) ---
_dummy_pos = [{'role': 'assistant', 'content': '', 'tool_calls': [{'type': 'function', 'function': {'name': 'search_x_twitter', 'arguments': {'expression': '1+1'}}}]}]
_dummy_neg = [{'role': 'assistant', 'content': 'PASS'}]
assert _reward_for_messages(_dummy_pos) == 1.0, 'positive case failed'
assert _reward_for_messages(_dummy_neg) == -1.0, 'negative case failed'
assert search_x_twitter_reward_func([], [_dummy_pos, _dummy_neg]) == [1.0, -1.0], 'batch func failed'
_m, _r = evaluate_reward([{'messages': _dummy_pos, 'persona': 'test'}, {'messages': _dummy_neg, 'persona': 'test'}], label='selftest')
assert _m == 0.0 and _r == 50.0, f'selftest stats wrong: {_m}, {_r}'
print('reward funcs ready: evaluate_reward, search_x_twitter_reward_func (selftest passed)' )


In [ ]:
'Training loop: rotating markets, chained adapters, Drive mirroring, held-out eval'
# Same machinery as the hardened overfit cell: adapter chaining, Drive sync + resume,
# TRL offline rollout with the cycling rollout_func, completion length capped to 4096,
# 20GiB GPU memory budget. Differences: markets ROTATE (rollout k takes the next
# disjoint slice of MARKETS_PER_ROLLOUT markets from a seeded shuffle of the train
# pool) and a fixed 20-market held-out eval set is reserved out of the pool.
import time, gc, json, os, random
from pathlib import Path
os.environ['TRL_EXPERIMENTAL_SILENCE'] = '1'   # silence the experimental rollout_func warning

N_TRAIN_ROLLOUTS = 20        # number of train+eval rollouts
max_steps = int((NUM_ITERATIONS*MARKETS_PER_ROLLOUT*NUM_GENERATIONS)/(GRAD_ACCUM_STEPS*PER_DEVICE_BATCH_SIZE))
print('max_steps per rollout: ' + str(max_steps) + ' (per_device=' + str(PER_DEVICE_BATCH_SIZE) + ', grad_accum=' + str(GRAD_ACCUM_STEPS) + ')')


# --- Fixed split of the pool (deterministic, no overlap):
#   - last N_EVAL_MARKETS unique markets      -> held-out eval (never trained on)
#   - the N_TRAIN_POOL_MARKETS before those   -> training pool (rotation divides it
#     exactly into len/N disjoint rollouts of MARKETS_PER_ROLLOUT, so consecutive
#     rollouts never share a market and nothing is skipped)
#   - any remainder at the front of the pool  -> discarded
N_EVAL_MARKETS = 20
N_TRAIN_POOL_MARKETS = 900
RUN_BASELINE_EVAL = True   # one extra eval before the first rollout (reference point)
EVAL_LOG_STEP = globals().get('EVAL_LOG_STEP', 0)   # global monotonic wandb step (kept across loop cells)
_market_cids = []
for _tp in task_pairs:
    if _tp[3] not in _market_cids:
        _market_cids.append(_tp[3])
assert len(_market_cids) >= N_EVAL_MARKETS + N_TRAIN_POOL_MARKETS, (
    'pool has only ' + str(len(_market_cids)) + ' unique markets, need at least ' +
    str(N_EVAL_MARKETS + N_TRAIN_POOL_MARKETS))
EVAL_CIDS = _market_cids[-N_EVAL_MARKETS:]
TRAIN_POOL_CIDS = _market_cids[len(_market_cids) - N_EVAL_MARKETS - N_TRAIN_POOL_MARKETS:
                               len(_market_cids) - N_EVAL_MARKETS]
assert len(TRAIN_POOL_CIDS) == N_TRAIN_POOL_MARKETS
assert len(TRAIN_POOL_CIDS) % MARKETS_PER_ROLLOUT == 0, (
    'train pool (' + str(len(TRAIN_POOL_CIDS)) + ') is not a multiple of MARKETS_PER_ROLLOUT (' +
    str(MARKETS_PER_ROLLOUT) + ') - rotation would overlap or skip markets')
_n_discarded = len(_market_cids) - N_EVAL_MARKETS - len(TRAIN_POOL_CIDS)
print('pool: ' + str(len(_market_cids)) + ' markets | train pool: ' + str(len(TRAIN_POOL_CIDS)) +
      ' | held-out eval: ' + str(len(EVAL_CIDS)) + ' (fixed, never trained on)' +
      ' | discarded: ' + str(_n_discarded) +
      ' | disjoint rollouts per epoch: ' + str(len(TRAIN_POOL_CIDS) // MARKETS_PER_ROLLOUT))
# --- Rotation: rollout k takes the next disjoint MARKETS_PER_ROLLOUT slice of a
# seeded shuffle of the train pool; the pool reshuffles (new seed) each epoch.
_per_epoch = max(1, len(TRAIN_POOL_CIDS) // MARKETS_PER_ROLLOUT)
def _markets_for_rollout(k):
    epoch = k // _per_epoch
    pos = (k % _per_epoch) * MARKETS_PER_ROLLOUT
    order = TRAIN_POOL_CIDS[:]
    random.Random(7000 + epoch).shuffle(order)
    return order[pos:pos + MARKETS_PER_ROLLOUT]
# Self-healing prerequisites (so this cell works even if 'Steps 5 and 6' was not run after a restart).
try:
    _chk = DRIVE_CACHE
except NameError:
    DRIVE_CACHE = '/root/.cache/huggingface/hub'
    print('DRIVE_CACHE was missing, defaulted to ' + DRIVE_CACHE)
try:
    _chk = quantization_config
except NameError:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_storage=torch.bfloat16)
    print('quantization_config was missing, recreated with NF4 defaults')
try:
    _chk = peft_config
except NameError:
    from peft import LoraConfig
    peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES, lora_dropout=LORA_DROPOUT, bias='none', task_type='CAUSAL_LM')
    print('peft_config was missing, recreated from LORA_* config')
try:
    _chk = search_x_twitter_reward_func
except NameError:
    raise RuntimeError('Cell 5c (reward funcs) must be run first')


# Drive persistence: mirror adapters + traces to Drive so a disconnected VM can resume.
DRIVE_OUTPUT_DIR = globals().get('DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/RL_supply_disk/grpo_output')

def _drive_ok():
    import os as _os
    return _os.path.ismount('/content/drive')

def _sync_to_drive(local_path):
    import os as _os, shutil as _sh, time as _tm
    if not _os.path.exists(local_path):
        return None
    if not _drive_ok():
        print('Drive not mounted, keeping local only: ' + str(local_path))
        return None
    try:
        rel = _os.path.relpath(local_path, OUTPUT_DIR)
        dest = _os.path.join(DRIVE_OUTPUT_DIR, rel)

        # Guard: source already lives on Drive (OUTPUT_DIR == DRIVE_OUTPUT_DIR)
        if _os.path.realpath(dest) == _os.path.realpath(local_path):
            print('already on Drive, nothing to sync: ' + str(local_path))
            return local_path

        _os.makedirs(_os.path.dirname(dest), exist_ok=True)

        if _os.path.isdir(local_path):
            tmp, old = dest + '.tmp', dest + '.old'

            # 1) Copy to a temp dir. dest is untouched while the copy can fail.
            _err = None
            for _attempt in range(4):
                try:
                    _sh.rmtree(tmp, ignore_errors=True)
                    _sh.copytree(local_path, tmp)
                    _err = None
                    break
                except Exception as _ce:
                    _err = _ce
                    _tm.sleep(5 * (_attempt + 1))
            if _err is not None:
                _sh.rmtree(tmp, ignore_errors=True)
                raise _err

            # 2) Verify the temp copy before swapping.
            def _tree_files(root):
                return sorted(_os.path.relpath(_os.path.join(d, f), root)
                              for d, _, fs in _os.walk(root) for f in fs)
            if _tree_files(tmp) != _tree_files(local_path):
                _sh.rmtree(tmp, ignore_errors=True)
                raise IOError('incomplete copy of ' + str(local_path))

            # 3) Swap. Old dest moves aside first; restored if the rename fails.
            _sh.rmtree(old, ignore_errors=True)
            _moved_old = False
            if _os.path.exists(dest):
                _os.rename(dest, old)
                _moved_old = True
            try:
                _os.rename(tmp, dest)
            except Exception:
                if _moved_old and not _os.path.exists(dest):
                    _os.rename(old, dest)
                raise
            _sh.rmtree(old, ignore_errors=True)
        else:
            tmp = dest + '.tmp'
            _err = None
            for _attempt in range(4):
                try:
                    _sh.copyfile(local_path, tmp)
                    _err = None
                    break
                except Exception as _ce:
                    _err = _ce
                    _tm.sleep(5 * (_attempt + 1))
            if _err is not None:
                raise _err
            _os.replace(tmp, dest)   # atomic on files

        print('Synced to Drive: ' + dest)
        return dest
    except Exception as e:
        print('Drive sync failed (' + str(e)[:200] + '), continuing locally')
        return None

def _newest_adapter(root):
    import os as _os, glob as _gg
    cands = sorted(_gg.glob(_os.path.join(root, 'train_iter_*', 'final_adapter')), key=lambda p: _os.path.getmtime(p))
    return cands[-1] if cands else None

if LORA_ADAPTER_PATH is None:
    _local_latest = _newest_adapter(OUTPUT_DIR)
    if _local_latest:
        LORA_ADAPTER_PATH = _local_latest
        print('Resuming rollouts from local adapter: ' + LORA_ADAPTER_PATH)
    elif _drive_ok():
        import os as _os2, shutil as _sh2
        _d_latest = _newest_adapter(DRIVE_OUTPUT_DIR)
        if _d_latest:
            _rel = _os2.path.relpath(_d_latest, DRIVE_OUTPUT_DIR)
            _local = _os2.path.join(OUTPUT_DIR, _rel)
            _os2.makedirs(_os2.path.dirname(_local), exist_ok=True)
            _sh2.copytree(_d_latest, _local, dirs_exist_ok=True)
            LORA_ADAPTER_PATH = _local
            print('Resumed from Drive adapter: ' + _d_latest)
    if LORA_ADAPTER_PATH is None:
        print('No prior adapter found, starting from base model')

async def _rl_loop():
    global LORA_ADAPTER_PATH, _last_train_traces
    eval_pairs = [tp for tp in task_pairs if tp[3] in set(EVAL_CIDS)]
    print('Eval pairs: ' + str(len(eval_pairs)) + ' (' + str(len(EVAL_CIDS)) + ' held-out markets x ' + str(NUM_GENERATIONS) + ' personas)')

    async def _eval_holdout(tag):
        """Fresh traces on the held-out markets with the CURRENT adapter - the real
        'are we learning' metric, independent of the markets we train on."""
        global EVAL_LOG_STEP
        if not eval_pairs:
            print('HOLDOUT ' + tag + ': no eval pairs, skipped')
            return None
        print('=== HOLDOUT EVAL ' + tag + ' adapter=' + str(LORA_ADAPTER_PATH))
        ensure_server(LORA_ADAPTER_PATH)
        t = time.time()
        eval_traces = await generate_traces_async(eval_pairs)
        globals()['_last_eval_traces'] = eval_traces
        print('eval rollout: ' + str(round(time.time() - t, 1)) + 's')
        try:
            Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
            with open(os.path.join(OUTPUT_DIR, 'train_eval_' + tag + '_traces.json'), 'w') as f:
                json.dump([{'condition_id': tr.get('condition_id'), 'persona': tr.get('persona'), 'messages': tr.get('messages')} for tr in eval_traces], f)
            _sync_to_drive(OUTPUT_DIR + '/train_eval_' + tag + '_traces.json')
        except Exception as e:
            print('eval trace checkpoint skipped: ' + str(e)[:200])
        m, r = evaluate_reward(eval_traces, label='HOLDOUT ' + tag)
        try:
            import wandb
            if wandb.run is not None:
                wandb.log({'eval/mean': m, 'eval/calc_rate': r}, step=EVAL_LOG_STEP)
                EVAL_LOG_STEP += 1
        except Exception:
            pass
        return m, r

    if RUN_BASELINE_EVAL:
        await _eval_holdout('baseline')

    for ro in range(N_TRAIN_ROLLOUTS):
        want = set(_markets_for_rollout(ro))
        batch_pairs = [tp for tp in task_pairs if tp[3] in want]
        print('--- train rollout ' + str(ro + 1) + '/' + str(N_TRAIN_ROLLOUTS) + ' markets=' + str(len(want)) + ' (rotating) adapter=' + str(LORA_ADAPTER_PATH))
        ensure_server(LORA_ADAPTER_PATH)
        t = time.time()
        traces = await generate_traces_async(batch_pairs)
        print('rollout: ' + str(round(time.time() - t, 1)) + 's')
        # Debugging checkpoint: keep traces alive after a failure so 30 min is not lost.
        try:
            _last_train_traces = traces
            Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
            with open(os.path.join(OUTPUT_DIR, f'train_ro{ro}_traces.json'), 'w') as f:
                json.dump([{'condition_id': tr.get('condition_id'), 'persona': tr.get('persona'), 'messages': tr.get('messages')} for tr in traces], f)
            print(f'saved {len(traces)}/{len(batch_pairs)} traces to {OUTPUT_DIR}/train_ro{ro}_traces.json (global _last_train_traces)')
            _sync_to_drive(OUTPUT_DIR + f'/train_ro{ro}_traces.json')
        except Exception as e:
            print(f'trace checkpoint skipped: {e}')
        pre_mean, pre_rate = evaluate_reward(traces, label='train ro' + str(ro + 1) + ' pre')
        t = time.time()
        compute_old_logprobs(traces)
        compute_ref_probs(traces)
        print('scoring: ' + str(round(time.time() - t, 1)) + 's')
        # Group by market and keep only full groups of G completions.
        traces_by_market = {}
        for tr in traces:
            traces_by_market.setdefault(tr['condition_id'], []).append(tr)
        # Debugging: show per-market counts before filtering.
        try:
            counts = {str(k)[:10]: len(v) for k, v in traces_by_market.items()}
            print(f'market groups: {len(traces_by_market)} counts={counts}')
        except Exception:
            pass
        precomputed = {}
        rollout_pos = {}
        dataset_rows = []
        for cid, mtraces in traces_by_market.items():
            if len(mtraces) < NUM_GENERATIONS:
                print('  skipping market ' + str(cid)[:10] + '...: only ' + str(len(mtraces)) + '/' + str(NUM_GENERATIONS))
                continue
            first = mtraces[0]
            key = (first['system_prompt'], first['market_context'])
            dataset_rows.append({'prompt': [{'role': 'system', 'content': first['system_prompt']}, {'role': 'user', 'content': first['market_context']}]})
            comps = []
            for trace in mtraces:
                full_ids, labels = mask_assistant_tokens(tok_local, trace['messages'])
                fl = full_ids.tolist()
                ll = labels.tolist()
                prompt_len = 0
                comp_ids = []
                for i, lbl in enumerate(ll):
                    if lbl != -100:
                        if not comp_ids:
                            prompt_len = i
                        comp_ids.append(fl[i])
                old_lp = trace['old_logprobs']
                ref_pb = trace['ref_probs']
                _cap = min(MAX_COMPLETION_LENGTH, 4096)
                _lp = [x['log_prob'] for x in old_lp][: _cap]
                _lt = [x['token_id'] for x in old_lp][: _cap]
                _rp = [x['prob'] for x in ref_pb][: _cap]
                _cc = comp_ids[: _cap]
                comps.append({'prompt_ids': fl[:prompt_len], 'completion_ids': _cc, 'logprobs': _lp, 'logprob_token_ids': _lt, 'ref_probs': _rp, 'completions': trace['messages']})
            precomputed[key] = comps
        print('dataset rows: ' + str(len(dataset_rows)) + '/' + str(len(want)) + ' (full groups only)')
        if len(dataset_rows) < len(want):
            print('WARNING: kept ' + str(len(dataset_rows)) + '/' + str(len(want)) + ' full groups. Groups with < ' + str(NUM_GENERATIONS) + ' traces were skipped.')
        from datasets import Dataset
        dataset = Dataset.from_list(dataset_rows)
        stop_server()
        def _prompt_key(p):
            try:
                if isinstance(p, dict) and 'prompt' in p:
                    p = p['prompt']
                if isinstance(p, list):
                    sys_c = ''
                    usr_c = ''
                    for m in p:
                        if not isinstance(m, dict):
                            continue
                        if m.get('role') == 'system' and not sys_c:
                            sys_c = m.get('content', '') or ''
                        if m.get('role') == 'user':
                            usr_c = m.get('content', '') or ''
                    if sys_c or usr_c:
                        return (sys_c, usr_c)
            except Exception:
                pass
            return (str(p),)
        def offline_rollout_func(prompts, trainer):
            out = {'prompt_ids': [], 'completion_ids': [], 'logprobs': [], 'logprob_token_ids': [], 'completions': []}
            for prompt in prompts:
                k = _prompt_key(prompt)
                if k not in precomputed:
                    raise KeyError('Prompt not found in overfit map. Available: ' + str(len(precomputed)))
                grp = precomputed[k]
                j = rollout_pos.get(k, 0) % len(grp)
                rollout_pos[k] = j + 1
                comp = grp[j]
                out['prompt_ids'].append(comp['prompt_ids'])
                out['completion_ids'].append(comp['completion_ids'])
                out['logprobs'].append(comp['logprobs'])
                out['logprob_token_ids'].append(comp['logprob_token_ids'])
                out['completions'].append(comp['completions'])
            # Debugging: confirm reward distribution on the exact batch the trainer sees.
            try:
                rs = search_x_twitter_reward_func(prompts, out['completions'])
                print(f'[offline_rollout] prompts={len(prompts)} completions={len(out["completions"])} mean={sum(rs)/len(rs):.3f} dist={{+1:{sum(1 for x in rs if x>0)}, -1:{sum(1 for x in rs if x<=0)}}}')
            except Exception as e:
                print(f'[offline_rollout] reward preview failed: {e}')
            return out
        from trl import GRPOTrainer, GRPOConfig
        # Your training model is already quantized (AWQ INT4), so BitsAndBytes NF4 would conflict.
        # Detect pre-quantization and train LoRA directly on the quantized weights.
        try:
            from transformers import AutoConfig
            _mcfg = AutoConfig.from_pretrained(TRAINING_MODEL, trust_remote_code=True)
            _preq = getattr(_mcfg, 'quantization_config', None)
        except Exception as _cfg_e:
            _preq = None
            print(f'model config check skipped: {_cfg_e}')
        _name_hit = any(k in TRAINING_MODEL.upper() for k in ['AWQ', 'GPTQ', 'INT4', 'INT8', 'FP4', 'COMPRESSED'])
        if _preq is not None or _name_hit:
            print(f'model {TRAINING_MODEL} is pre-quantized ({str(_preq)[:160]}), using quantization_config=None')
            _eff_quant = None
        else:
            _eff_quant = quantization_config
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        # L4 22GB fit: cap completion length and GPU memory (global 50k would OOM on 9B).
        _overfit_max_len = min(MAX_COMPLETION_LENGTH, 4096)
        print(f'train len={_overfit_max_len} (capped from {MAX_COMPLETION_LENGTH} for 22GB GPU)')
        args = GRPOConfig(output_dir=OUTPUT_DIR + '/train_iter_' + str(ro), per_device_train_batch_size=PER_DEVICE_BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM_STEPS, num_iterations=NUM_ITERATIONS, num_generations=NUM_GENERATIONS, max_completion_length=_overfit_max_len, learning_rate=LEARNING_RATE, beta=BETA, epsilon=EPSILON, loss_type=LOSS_TYPE, scale_rewards=SCALE_REWARDS, max_steps=max_steps, logging_steps=1, save_steps=50, bf16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False}, remove_unused_columns=False, model_init_kwargs={'trust_remote_code': True, 'device_map': 'auto', 'max_memory': {0: '20GiB', 'cpu': '120GiB'}, 'cache_dir': DRIVE_CACHE})
        t = time.time()
        trainer = GRPOTrainer(model=TRAINING_MODEL, reward_funcs=search_x_twitter_reward_func, args=args, train_dataset=dataset, quantization_config=_eff_quant, peft_config=peft_config, rollout_func=offline_rollout_func)
        if LORA_ADAPTER_PATH:
            try:
                import glob as _glob
                _cands = sorted(_glob.glob(os.path.join(LORA_ADAPTER_PATH, 'adapter_model.safetensors')) + _glob.glob(os.path.join(LORA_ADAPTER_PATH, 'adapter_model.bin')))
                assert _cands, 'no adapter weight file in ' + str(LORA_ADAPTER_PATH)
                if _cands[0].endswith('.safetensors'):
                    from safetensors.torch import load_file as _loadf
                    _state = _loadf(_cands[0])
                else:
                    _state = torch.load(_cands[0], map_location='cpu')
                from peft import set_peft_model_state_dict as _set_pm
                _set_pm(trainer.model, _state)
                print('chained training from previous adapter: ' + str(LORA_ADAPTER_PATH))
            except Exception as _chain_e:
                print('adapter chaining skipped (' + str(_chain_e)[:200] + '), training from base')
        trainer.train()
        print('training: ' + str(round(time.time() - t, 1)) + 's (batch x ' + str(NUM_ITERATIONS) + ' iters)')
        adapter_path = OUTPUT_DIR + '/train_iter_' + str(ro) + '/final_adapter'
        os.makedirs(adapter_path, exist_ok=True)
        trainer.save_model(adapter_path)
        _sync_to_drive(adapter_path)
        post_mean, post_rate = evaluate_reward(traces, label='train ro' + str(ro + 1) + ' post')
        print('reward ' + str(round(pre_mean, 4)) + ' -> ' + str(round(post_mean, 4)))
        try:
            import wandb
            if wandb.run is not None:
                wandb.log({'train/ro': ro, 'train/pre_mean': pre_mean, 'train/pre_rate': pre_rate}, step=ro)
        except Exception:
            pass
        del trainer
        gc.collect()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        LORA_ADAPTER_PATH = adapter_path
        print('next adapter: ' + adapter_path)
        # Held-out eval with the freshly trained adapter: is the policy actually learning?
        await _eval_holdout('ro' + str(ro + 1))
    print('done. final adapter: ' + str(LORA_ADAPTER_PATH))

print('Training loop ready: run await _rl_loop() manually when you are ready')
print('Note: Cell 5c (reward funcs) and the TRL patch cell must run first. Traces are checkpointed to OUTPUT_DIR/train_ro*_traces.json')
# To start the run: await _rl_loop()


In [ ]:
await _rl_loop()